# Policy Gradient & REINFORCE for Image Enhancement

## Module 6.2 — Deep RL: Policy Gradient Methods

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_06_Deep_RL/02_Policy_Gradient_REINFORCE/policy_gradient_reinforce.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:

1. **Derive** the Policy Gradient Theorem from first principles
2. **Understand** the score function estimator (REINFORCE)
3. **Implement** variance reduction using baselines
4. **Build** a policy gradient agent for image enhancement
5. **Train** and visualize policy learning dynamics

In [ ]:
# Install dependencies (Colab-compatible)
!pip install torch torchvision numpy matplotlib pillow scipy --quiet

In [ ]:
# Download REAL open-source dataset for Deep RL image environment
import torchvision
import numpy as np

# CIFAR-10 as our image environment (real photos to enhance)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = np.array([np.array(cifar10[i][0]) for i in range(500)])
print(f"✅ CIFAR-10 loaded: {len(real_images)} real photos as RL environment images")
print(f"   Shape: {real_images[0].shape} (32x32 RGB)")
print("   Agent will learn to enhance these REAL photographs!")

# Pre-trained model weights (for feature extraction)
print("✅ Pre-trained ResNet18 weights available for state encoding")

## Deep Derivation: REINFORCE and the Policy Gradient Theorem

### Step 1: Objective — Maximize Expected Return
$$J(\theta) = E_{\tau \sim \pi_\theta}\left[\sum_{t=0}^T \gamma^t r_t\right] = E_{\tau \sim \pi_\theta}[R(\tau)]$$

where trajectory $\tau = (s_0, a_0, r_0, s_1, a_1, r_1, \ldots)$.

### Step 2: Trajectory Probability
$$P(\tau | \theta) = \rho(s_0) \prod_{t=0}^{T-1} \pi_\theta(a_t | s_t) \cdot P(s_{t+1} | s_t, a_t)$$

### Step 3: Policy Gradient Derivation (The Log-Derivative Trick)
$$\nabla_\theta J(\theta) = \nabla_\theta \sum_\tau P(\tau|\theta) R(\tau)$$
$$= \sum_\tau \nabla_\theta P(\tau|\theta) R(\tau)$$
$$= \sum_\tau P(\tau|\theta) \frac{\nabla_\theta P(\tau|\theta)}{P(\tau|\theta)} R(\tau)$$

**Key identity (log-derivative trick):** $\frac{\nabla_\theta P(\tau|\theta)}{P(\tau|\theta)} = \nabla_\theta \log P(\tau|\theta)$

**Proof:**
$$\nabla_\theta \log P(\tau|\theta) = \frac{1}{P(\tau|\theta)} \nabla_\theta P(\tau|\theta) \quad \blacksquare$$

Therefore: $\nabla_\theta J(\theta) = E_{\tau \sim \pi_\theta}\left[\nabla_\theta \log P(\tau|\theta) \cdot R(\tau)\right]$

### Step 4: Simplifying $\nabla_\theta \log P(\tau|\theta)$
$$\log P(\tau|\theta) = \log \rho(s_0) + \sum_{t=0}^{T-1}\left[\log \pi_\theta(a_t|s_t) + \log P(s_{t+1}|s_t,a_t)\right]$$

Taking gradient w.r.t. $\theta$ (transition dynamics $P$ and $\rho$ don't depend on $\theta$):
$$\nabla_\theta \log P(\tau|\theta) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) \quad \blacksquare$$

**This is remarkable:** We don't need to know the environment dynamics!

### Step 5: REINFORCE Estimator
$$\nabla_\theta J(\theta) = E_{\tau}\left[\sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot R(\tau)\right]$$

Monte Carlo estimate with $N$ trajectories:
$$\hat{g} = \frac{1}{N}\sum_{i=1}^N \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t^{(i)}|s_t^{(i)}) \cdot R(\tau^{(i)})$$

### Step 6: Variance Reduction — Baseline
Subtracting baseline $b(s_t)$ from returns:
$$\hat{g} = \frac{1}{N}\sum_{i=1}^N \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot (R_t - b(s_t))$$

**Proof baseline doesn't bias the gradient:**
$$E\left[\nabla_\theta \log \pi_\theta(a|s) \cdot b(s)\right] = b(s) \sum_a \nabla_\theta \pi_\theta(a|s) = b(s) \cdot \nabla_\theta \underbrace{\sum_a \pi_\theta(a|s)}_{=1} = 0 \quad \blacksquare$$

**Optimal baseline** (minimizes variance): $b^*(s) = \frac{E[\|\nabla_\theta \log \pi\|^2 \cdot R]}{E[\|\nabla_\theta \log \pi\|^2]}$

In practice: $b(s_t) \approx V(s_t)$, so $R_t - b(s_t) \approx A(s_t, a_t)$.

### Step 7: Convergence Rate
REINFORCE converges at rate $O(1/\sqrt{T})$ to a local optimum, with variance $\text{Var}[\hat{g}] \propto \frac{1}{N}$.

**Why high variance?** Each trajectory samples one path through the MDP — returns can vary enormously between episodes. This motivates Actor-Critic methods (Module 06.03).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)

---

## 1. Policy Gradient Theorem — Full Derivation

### 1.1 The Objective

We want to directly optimize a parameterized policy $\pi_\theta(a|s)$ to maximize expected cumulative reward:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \gamma^t r_t\right] = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$$

where $\tau = (s_0, a_0, r_0, s_1, a_1, r_1, \ldots)$ is a trajectory sampled under policy $\pi_\theta$.

### 1.2 Trajectory Distribution

The probability of a trajectory under policy $\pi_\theta$:

$$p(\tau | \theta) = p(s_0) \prod_{t=0}^{T-1} \pi_\theta(a_t | s_t) \cdot p(s_{t+1} | s_t, a_t)$$

Therefore:

$$J(\theta) = \int p(\tau | \theta) R(\tau) \, d\tau$$

### 1.3 The Gradient — Log-Derivative Trick

Taking the gradient:

$$\nabla_\theta J(\theta) = \int \nabla_\theta p(\tau | \theta) R(\tau) \, d\tau$$

Using the **log-derivative trick**: $\nabla_\theta p(\tau|\theta) = p(\tau|\theta) \nabla_\theta \log p(\tau|\theta)$:

$$\nabla_\theta J(\theta) = \int p(\tau | \theta) \nabla_\theta \log p(\tau | \theta) \cdot R(\tau) \, d\tau$$

$$= \mathbb{E}_{\tau \sim \pi_\theta}\left[\nabla_\theta \log p(\tau | \theta) \cdot R(\tau)\right]$$

### 1.4 Simplification — Removing Dynamics

Expanding $\log p(\tau | \theta)$:

$$\log p(\tau | \theta) = \log p(s_0) + \sum_{t=0}^{T-1} \left[\log \pi_\theta(a_t | s_t) + \log p(s_{t+1} | s_t, a_t)\right]$$

Taking the gradient (dynamics terms vanish since they don't depend on $\theta$):

$$\nabla_\theta \log p(\tau | \theta) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t | s_t)$$

### 1.5 The Policy Gradient Theorem

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot G_t\right]}$$

where $G_t = \sum_{k=t}^{T-1} \gamma^{k-t} r_k$ is the **return** (reward-to-go) from time step $t$.

**Key insight**: We use $G_t$ instead of $R(\tau)$ because actions at time $t$ cannot affect past rewards (causality).

---

## 2. REINFORCE Algorithm

### 2.1 Monte Carlo Policy Gradient

REINFORCE uses Monte Carlo sampling to estimate the gradient:

$$\hat{g} = \frac{1}{N} \sum_{i=1}^{N} \sum_{t=0}^{T_i - 1} \nabla_\theta \log \pi_\theta(a_t^{(i)} | s_t^{(i)}) \cdot G_t^{(i)}$$

With a single trajectory ($N=1$):

$$\hat{g} = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot G_t$$

### 2.2 Algorithm Pseudocode

1. Initialize policy parameters $\theta$
2. **For** each episode:
   - Generate trajectory $\tau = (s_0, a_0, r_0, \ldots, s_T)$ using $\pi_\theta$
   - **For** each step $t$:
     - Compute return $G_t = \sum_{k=t}^{T-1} \gamma^{k-t} r_k$
   - Update: $\theta \leftarrow \theta + \alpha \sum_{t} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t$

### 2.3 Intuition

- If $G_t > 0$: Increase probability of action $a_t$ in state $s_t$ (reinforce good actions)
- If $G_t < 0$: Decrease probability of action $a_t$ in state $s_t$ (discourage bad actions)
- The magnitude of the update is proportional to the return

---

## 3. Variance Reduction with Baselines

### 3.1 The High-Variance Problem

The REINFORCE estimator has high variance:

$$\text{Var}[\hat{g}] = \text{Var}\left[\nabla_\theta \log \pi_\theta(a|s) \cdot G_t\right]$$

Since $G_t$ can vary dramatically between episodes, the gradient estimate is noisy.

### 3.2 Baseline Subtraction

Subtract a state-dependent baseline $b(s_t)$ from the return:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[\sum_{t} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot (G_t - b(s_t))\right]$$

**Theorem**: This is still an unbiased estimator for any $b(s)$ that doesn't depend on $a$:

$$\mathbb{E}_{a \sim \pi_\theta}\left[\nabla_\theta \log \pi_\theta(a|s) \cdot b(s)\right] = b(s) \cdot \nabla_\theta \sum_a \pi_\theta(a|s) = b(s) \cdot \nabla_\theta 1 = 0$$

### 3.3 Optimal Baseline

The variance-minimizing baseline is:

$$b^*(s) = \frac{\mathbb{E}\left[\|\nabla_\theta \log \pi_\theta(a|s)\|^2 \cdot G_t\right]}{\mathbb{E}\left[\|\nabla_\theta \log \pi_\theta(a|s)\|^2\right]}$$

In practice, we approximate with the **value function** $V^\pi(s) \approx \mathbb{E}[G_t | s_t = s]$, which is close to optimal.

### 3.4 Variance Reduction Factor

With baseline $b = V^\pi(s)$, the effective signal becomes the **advantage**:

$$A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s)$$

The variance reduction can be quantified:

$$\frac{\text{Var}[\hat{g}_{\text{with baseline}}]}{\text{Var}[\hat{g}_{\text{no baseline}}]} \approx \frac{\text{Var}[A^\pi(s,a)]}{\text{Var}[G_t]} \ll 1$$

In [ ]:
# Demonstrate variance reduction with baselines

np.random.seed(42)

# Simulate returns from a simple environment
n_samples = 1000
returns = np.random.normal(10, 5, n_samples)  # Mean 10, std 5
log_probs = np.random.normal(0, 1, n_samples)  # Simulated log-prob gradients

# Without baseline
gradients_no_baseline = log_probs * returns

# With mean baseline
baseline_mean = np.mean(returns)
gradients_mean_baseline = log_probs * (returns - baseline_mean)

# With running average baseline
running_baseline = np.cumsum(returns) / (np.arange(n_samples) + 1)
gradients_running_baseline = log_probs * (returns - running_baseline)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(gradients_no_baseline, bins=50, alpha=0.7, color='red', density=True)
axes[0].set_title(f'No Baseline\nVar = {np.var(gradients_no_baseline):.2f}', fontsize=11)
axes[0].set_xlabel('Gradient Estimate')
axes[0].axvline(x=0, color='black', linestyle='--', alpha=0.5)

axes[1].hist(gradients_mean_baseline, bins=50, alpha=0.7, color='blue', density=True)
axes[1].set_title(f'Mean Baseline (b={baseline_mean:.1f})\nVar = {np.var(gradients_mean_baseline):.2f}', fontsize=11)
axes[1].set_xlabel('Gradient Estimate')
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.5)

axes[2].hist(gradients_running_baseline, bins=50, alpha=0.7, color='green', density=True)
axes[2].set_title(f'Running Baseline\nVar = {np.var(gradients_running_baseline):.2f}', fontsize=11)
axes[2].set_xlabel('Gradient Estimate')
axes[2].axvline(x=0, color='black', linestyle='--', alpha=0.5)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('Variance Reduction via Baseline Subtraction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Variance reduction with mean baseline: {np.var(gradients_no_baseline)/np.var(gradients_mean_baseline):.1f}x")

## Deep Dive: Variance Analysis and Optimal Baseline Derivation

### The Variance Problem in REINFORCE

The REINFORCE gradient estimator $\hat{g} = \nabla_\theta \log \pi_\theta(a|s) \cdot G_t$ is **unbiased** but has **high variance**. Understanding and reducing this variance is the central challenge of policy gradient methods.

### Full Variance Decomposition

For a single-step gradient estimate $\hat{g} = \nabla_\theta \log \pi_\theta(a|s) \cdot (G_t - b)$:

$$\text{Var}[\hat{g}] = \mathbb{E}\left[\|\hat{g}\|^2\right] - \left\|\mathbb{E}[\hat{g}]\right\|^2$$

Since the estimator is unbiased regardless of $b$, $\|\mathbb{E}[\hat{g}]\|^2$ is constant. Minimizing variance reduces to minimizing:

$$\mathbb{E}\left[\|\hat{g}\|^2\right] = \mathbb{E}\left[\|\nabla_\theta \log \pi_\theta(a|s)\|^2 \cdot (G_t - b)^2\right]$$

### Derivation of the Optimal Baseline

**Step 1:** Write the second moment as a function of $b$:

$$f(b) = \mathbb{E}\left[\|\nabla_\theta \log \pi_\theta(a|s)\|^2 \cdot (G_t - b)^2\right]$$

**Step 2:** Take the derivative with respect to $b$ and set to zero:

$$\frac{df}{db} = -2\mathbb{E}\left[\|\nabla_\theta \log \pi_\theta(a|s)\|^2 \cdot (G_t - b)\right] = 0$$

**Step 3:** Solve for $b$:

$$b \cdot \mathbb{E}\left[\|\nabla_\theta \log \pi_\theta(a|s)\|^2\right] = \mathbb{E}\left[\|\nabla_\theta \log \pi_\theta(a|s)\|^2 \cdot G_t\right]$$

$$\boxed{b^*(s) = \frac{\mathbb{E}\left[\|\nabla_\theta \log \pi_\theta(a|s)\|^2 \cdot G_t\right]}{\mathbb{E}\left[\|\nabla_\theta \log \pi_\theta(a|s)\|^2\right]}}$$

**Step 4 — Verify this is a minimum:** $\frac{d^2 f}{db^2} = 2\mathbb{E}[\|\nabla_\theta \log \pi_\theta\|^2] > 0$, confirming $b^*$ is a minimum. $\blacksquare$

### Interpretation

The optimal baseline is a **weighted average** of returns, where the weights are the squared gradient magnitudes $\|\nabla_\theta \log \pi_\theta(a|s)\|^2$. States and actions where the policy gradient is large receive more weight.

**Why $V^\pi(s)$ is a good approximation:** If $\|\nabla_\theta \log \pi_\theta(a|s)\|^2$ is approximately constant across actions (which happens when the policy is close to uniform), then:

$$b^*(s) \approx \frac{\mathbb{E}[G_t]}{\mathbb{E}[1]} = \mathbb{E}[G_t | s_t = s] = V^\pi(s)$$

### Quantifying Variance Reduction

With the optimal baseline vs. no baseline, the variance reduction ratio is:

$$\frac{\text{Var}[\hat{g}_{b^*}]}{\text{Var}[\hat{g}_{b=0}]} = 1 - \frac{\left(\mathbb{E}\left[\|\nabla \log \pi\|^2 G_t\right]\right)^2}{\mathbb{E}\left[\|\nabla \log \pi\|^2\right] \cdot \mathbb{E}\left[\|\nabla \log \pi\|^2 G_t^2\right]}$$

By the Cauchy-Schwarz inequality, this ratio is always $\leq 1$, confirming the baseline never increases variance. In practice, the reduction can be **10x–100x**, making the difference between convergence and failure.

---

## 4. Image Enhancement Environment

In [ ]:
class ImageEnhancementEnv:
    """Environment for policy-gradient-based image enhancement."""

    ACTIONS = [
        'sharpen_light', 'sharpen_strong', 'denoise_light', 'denoise_strong',
        'brighten', 'darken', 'contrast_up', 'contrast_down',
        'gamma_up', 'gamma_down', 'no_op'
    ]

    def __init__(self, image_size=48, max_steps=8):
        self.image_size = image_size
        self.max_steps = max_steps
        self.n_actions = len(self.ACTIONS)
        self.reset()

    def _generate_target(self):
        img = np.zeros((3, self.image_size, self.image_size), dtype=np.float32)
        x = np.linspace(0, 6*np.pi, self.image_size)
        y = np.linspace(0, 6*np.pi, self.image_size)
        X, Y = np.meshgrid(x, y)
        img[0] = 0.5 + 0.4 * np.sin(X) * np.cos(Y * 0.5)
        img[1] = 0.4 + 0.3 * np.sin(X + Y)
        img[2] = 0.6 + 0.2 * np.cos(2 * X - Y)
        return np.clip(img, 0, 1)

    def _degrade(self, img):
        degraded = img.copy()
        degraded += np.random.normal(0, 0.12, img.shape).astype(np.float32)
        degraded = 0.5 + 0.55 * (degraded - 0.5)
        degraded += np.random.uniform(-0.08, 0.08)
        return np.clip(degraded, 0, 1)

    def _psnr(self, img1, img2):
        mse = np.mean((img1 - img2) ** 2)
        if mse < 1e-10:
            return 50.0
        return 10 * np.log10(1.0 / mse)

    def reset(self):
        self.step_count = 0
        self.target = self._generate_target()
        self.current = self._degrade(self.target)
        self.prev_psnr = self._psnr(self.current, self.target)
        return self.current.copy()

    def step(self, action):
        self.step_count += 1
        action_name = self.ACTIONS[action]
        img = self.current.copy()

        if action_name == 'sharpen_light':
            k = np.array([[0, -0.3, 0], [-0.3, 2.2, -0.3], [0, -0.3, 0]])
            for c in range(3):
                img[c] = convolve(img[c], k, mode='reflect')
        elif action_name == 'sharpen_strong':
            k = np.array([[0, -0.5, 0], [-0.5, 3.0, -0.5], [0, -0.5, 0]])
            for c in range(3):
                img[c] = convolve(img[c], k, mode='reflect')
        elif action_name == 'denoise_light':
            k = np.ones((3, 3)) / 9.0
            for c in range(3):
                img[c] = convolve(img[c], k, mode='reflect')
        elif action_name == 'denoise_strong':
            k = np.ones((5, 5)) / 25.0
            for c in range(3):
                img[c] = convolve(img[c], k, mode='reflect')
        elif action_name == 'brighten':
            img += 0.04
        elif action_name == 'darken':
            img -= 0.04
        elif action_name == 'contrast_up':
            img = 0.5 + 1.15 * (img - 0.5)
        elif action_name == 'contrast_down':
            img = 0.5 + 0.85 * (img - 0.5)
        elif action_name == 'gamma_up':
            img = np.power(np.clip(img, 0, 1), 0.9)
        elif action_name == 'gamma_down':
            img = np.power(np.clip(img, 0, 1), 1.1)

        self.current = np.clip(img, 0, 1)
        new_psnr = self._psnr(self.current, self.target)
        reward = (new_psnr - self.prev_psnr)
        self.prev_psnr = new_psnr
        done = self.step_count >= self.max_steps

        return self.current.copy(), reward, done, {'psnr': new_psnr}


# Test
env = ImageEnhancementEnv()
s = env.reset()
print(f"State shape: {s.shape}, Actions: {env.n_actions}")
print(f"Initial PSNR: {env.prev_psnr:.2f} dB")

---

## 5. Policy Network Architecture

In [ ]:
class PolicyNetwork(nn.Module):
    """CNN-based policy network that outputs action probabilities."""

    def __init__(self, n_actions, image_size=48):
        super(PolicyNetwork, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4)
        )

        self.policy_head = nn.Sequential(
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, n_actions)
        )

    def forward(self, x):
        features = self.features(x)
        features = features.view(features.size(0), -1)
        logits = self.policy_head(features)
        return F.softmax(logits, dim=-1)


class ValueNetwork(nn.Module):
    """Baseline value network V(s) for variance reduction."""

    def __init__(self, image_size=48):
        super(ValueNetwork, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4)
        )

        self.value_head = nn.Sequential(
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        features = self.features(x)
        features = features.view(features.size(0), -1)
        return self.value_head(features).squeeze(-1)


# Instantiate
policy_net = PolicyNetwork(env.n_actions, image_size=48).to(device)
value_net = ValueNetwork(image_size=48).to(device)

print(f"Policy Network parameters: {sum(p.numel() for p in policy_net.parameters()):,}")
print(f"Value Network parameters: {sum(p.numel() for p in value_net.parameters()):,}")

# Test
test_state = torch.randn(1, 3, 48, 48).to(device)
probs = policy_net(test_state)
value = value_net(test_state)
print(f"\nAction probabilities: {probs.detach().cpu().numpy().round(3)}")
print(f"State value: {value.item():.4f}")

## Deep Dive: Natural Policy Gradient and the Fisher Information Matrix

### Motivation — The Problem with Vanilla Gradients

The standard policy gradient $\nabla_\theta J(\theta)$ operates in **parameter space**. But a small change in $\theta$ can cause a large or small change in the policy $\pi_\theta$, depending on the parameterization. This makes the learning rate difficult to tune.

**Example:** For a softmax policy $\pi_\theta(a|s) \propto e^{\theta_a}$:
- Near $\theta = \mathbf{0}$: all actions equally likely, small $\Delta\theta$ → large policy change
- When $|\theta_a| \gg 1$: policy nearly deterministic, large $\Delta\theta$ → tiny policy change

### The Fisher Information Matrix

The **Fisher information matrix** (FIM) measures the curvature of the log-likelihood surface:

$$\mathbf{F}_\theta = \mathbb{E}_{s \sim d^{\pi_\theta}, a \sim \pi_\theta(\cdot|s)} \left[\nabla_\theta \log \pi_\theta(a|s) \cdot \nabla_\theta \log \pi_\theta(a|s)^T\right]$$

**Key property — FIM equals the Hessian of KL divergence:**

$$\mathbf{F}_\theta = \nabla_\theta^2 D_{\text{KL}}(\pi_\theta \| \pi_{\theta'}) \Big|_{\theta' = \theta}$$

**Proof:** The KL divergence between $\pi_\theta$ and $\pi_{\theta + \Delta\theta}$ to second order is:

$$D_{\text{KL}}(\pi_\theta \| \pi_{\theta + \Delta\theta}) \approx \frac{1}{2} \Delta\theta^T \mathbf{F}_\theta \, \Delta\theta$$

This follows because $D_{\text{KL}}(\pi_\theta \| \pi_\theta) = 0$ (first-order term vanishes at minimum), and the Hessian of KL at $\theta' = \theta$ is exactly the Fisher information. $\blacksquare$

### The Natural Gradient

The **natural policy gradient** (Kakade, 2001) replaces the standard gradient with:

$$\tilde{\nabla}_\theta J(\theta) = \mathbf{F}_\theta^{-1} \nabla_\theta J(\theta)$$

**Interpretation:** This is the steepest ascent direction in **policy space** (measured by KL divergence) rather than in parameter space (measured by Euclidean distance).

**Derivation via constrained optimization:**

$$\tilde{\nabla}_\theta J = \arg\max_{\mathbf{d}} \; \mathbf{d}^T \nabla_\theta J(\theta) \quad \text{s.t.} \quad \frac{1}{2}\mathbf{d}^T \mathbf{F}_\theta \mathbf{d} \leq \epsilon$$

Using Lagrange multipliers: $\nabla_\theta J - \lambda \mathbf{F}_\theta \mathbf{d} = 0 \implies \mathbf{d} \propto \mathbf{F}_\theta^{-1} \nabla_\theta J$. $\blacksquare$

### The Natural Gradient Update

$$\theta_{k+1} = \theta_k + \alpha \mathbf{F}_{\theta_k}^{-1} \nabla_\theta J(\theta_k)$$

**Properties:**
1. **Parameterization invariant:** If $\theta' = g(\theta)$ for some diffeomorphism $g$, the natural gradient update in $\theta'$-space is equivalent to the update in $\theta$-space.
2. **Fisher-efficient:** Achieves the Cramér-Rao lower bound for estimation efficiency.
3. **Connects to TRPO/PPO:** TRPO approximately computes the natural gradient via conjugate gradient, and PPO approximates it via clipping.

### Computational Cost

The FIM is a $p \times p$ matrix where $p = |\theta|$. For neural networks with millions of parameters:
- Storing $\mathbf{F}$ requires $O(p^2)$ memory — infeasible
- Inverting $\mathbf{F}$ costs $O(p^3)$ — prohibitive
- **K-FAC** (Kronecker-Factored Approximate Curvature) approximates $\mathbf{F}^{-1}$ efficiently
- TRPO uses **conjugate gradient** to compute $\mathbf{F}^{-1}\mathbf{g}$ without forming $\mathbf{F}$ explicitly
- PPO avoids computing $\mathbf{F}$ entirely by using clipping as a proxy

This progression — Natural Gradient → TRPO → PPO — represents increasing practical simplification of the same underlying idea: constrain policy updates in distribution space.

---

## 6. REINFORCE Implementation

### Loss Function

The policy gradient loss (to maximize via gradient ascent, we minimize the negative):

$$\mathcal{L}_{\text{policy}} = -\frac{1}{T} \sum_{t=0}^{T-1} \log \pi_\theta(a_t | s_t) \cdot (G_t - b(s_t))$$

The baseline (value function) loss:

$$\mathcal{L}_{\text{value}} = \frac{1}{T} \sum_{t=0}^{T-1} (V_\phi(s_t) - G_t)^2$$

In [ ]:
class REINFORCEAgent:
    """REINFORCE with learned baseline for image enhancement."""

    def __init__(self, n_actions, image_size=48, lr_policy=3e-4,
                 lr_value=1e-3, gamma=0.95, use_baseline=True):
        self.gamma = gamma
        self.use_baseline = use_baseline

        self.policy_net = PolicyNetwork(n_actions, image_size).to(device)
        self.value_net = ValueNetwork(image_size).to(device) if use_baseline else None

        self.policy_optimizer = optim.Adam(self.policy_net.parameters(), lr=lr_policy)
        if use_baseline:
            self.value_optimizer = optim.Adam(self.value_net.parameters(), lr=lr_value)

        # Episode storage
        self.saved_log_probs = []
        self.saved_states = []
        self.rewards = []

        # Metrics
        self.policy_losses = []
        self.value_losses = []
        self.entropies = []

    def select_action(self, state):
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs = self.policy_net(state_tensor)
        dist = Categorical(probs)
        action = dist.sample()

        self.saved_log_probs.append(dist.log_prob(action))
        self.saved_states.append(state_tensor)
        self.entropies.append(dist.entropy().item())

        return action.item()

    def compute_returns(self):
        """Compute discounted returns G_t for each timestep."""
        returns = []
        G = 0
        for r in reversed(self.rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        # Normalize returns for stable training
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        return returns

    def update(self):
        """Update policy (and value) networks using collected trajectory."""
        returns = self.compute_returns()
        log_probs = torch.stack(self.saved_log_probs)

        if self.use_baseline:
            states = torch.cat(self.saved_states, dim=0)
            values = self.value_net(states)
            advantages = returns - values.detach()

            # Policy loss with baseline
            policy_loss = -(log_probs * advantages).mean()

            # Value loss
            value_loss = F.mse_loss(values, returns)

            # Update value network
            self.value_optimizer.zero_grad()
            value_loss.backward()
            self.value_optimizer.step()
            self.value_losses.append(value_loss.item())
        else:
            policy_loss = -(log_probs * returns).mean()

        # Update policy network
        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=1.0)
        self.policy_optimizer.step()

        self.policy_losses.append(policy_loss.item())

        # Clear episode data
        self.saved_log_probs = []
        self.saved_states = []
        self.rewards = []

        return policy_loss.item()


print("REINFORCEAgent defined successfully.")

---

## 7. Training — Comparison With and Without Baseline

In [ ]:
def train_reinforce(use_baseline=True, n_episodes=400, label=""):
    """Train REINFORCE agent."""
    env = ImageEnhancementEnv(image_size=48, max_steps=8)
    agent = REINFORCEAgent(
        n_actions=env.n_actions,
        image_size=48,
        lr_policy=3e-4,
        lr_value=1e-3,
        gamma=0.95,
        use_baseline=use_baseline
    )

    episode_rewards = []
    episode_psnrs = []

    for episode in range(n_episodes):
        state = env.reset()
        ep_reward = 0

        for t in range(env.max_steps):
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            agent.rewards.append(reward)
            state = next_state
            ep_reward += reward
            if done:
                break

        agent.update()
        episode_rewards.append(ep_reward)
        episode_psnrs.append(info['psnr'])

        if (episode + 1) % 100 == 0:
            avg_r = np.mean(episode_rewards[-50:])
            avg_psnr = np.mean(episode_psnrs[-50:])
            print(f"[{label}] Ep {episode+1:4d} | Avg Reward: {avg_r:.3f} | PSNR: {avg_psnr:.2f} dB")

    return agent, episode_rewards, episode_psnrs


# Train both variants
print("=" * 60)
print("Training REINFORCE WITH baseline...")
print("=" * 60)
agent_baseline, rewards_baseline, psnrs_baseline = train_reinforce(
    use_baseline=True, n_episodes=400, label="Baseline"
)

print("\n" + "=" * 60)
print("Training REINFORCE WITHOUT baseline...")
print("=" * 60)
agent_no_baseline, rewards_no_baseline, psnrs_no_baseline = train_reinforce(
    use_baseline=False, n_episodes=400, label="No Baseline"
)

In [ ]:
# Compare training curves

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

window = 30

# Rewards comparison
ax = axes[0, 0]
ax.plot(rewards_baseline, alpha=0.2, color='blue')
ax.plot(rewards_no_baseline, alpha=0.2, color='red')
if len(rewards_baseline) >= window:
    sm_b = np.convolve(rewards_baseline, np.ones(window)/window, mode='valid')
    sm_nb = np.convolve(rewards_no_baseline, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(rewards_baseline)), sm_b, color='blue', linewidth=2, label='With Baseline')
    ax.plot(range(window-1, len(rewards_no_baseline)), sm_nb, color='red', linewidth=2, label='Without Baseline')
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward')
ax.set_title('Reward Comparison: Baseline Effect', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# PSNR comparison
ax = axes[0, 1]
if len(psnrs_baseline) >= window:
    sm_pb = np.convolve(psnrs_baseline, np.ones(window)/window, mode='valid')
    sm_pnb = np.convolve(psnrs_no_baseline, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(psnrs_baseline)), sm_pb, color='blue', linewidth=2, label='With Baseline')
    ax.plot(range(window-1, len(psnrs_no_baseline)), sm_pnb, color='red', linewidth=2, label='Without Baseline')
ax.set_xlabel('Episode')
ax.set_ylabel('PSNR (dB)')
ax.set_title('Image Quality (PSNR) Comparison', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Policy loss
ax = axes[1, 0]
ax.plot(agent_baseline.policy_losses, alpha=0.5, color='blue', label='With Baseline')
ax.plot(agent_no_baseline.policy_losses, alpha=0.5, color='red', label='Without Baseline')
ax.set_xlabel('Update Step')
ax.set_ylabel('Policy Loss')
ax.set_title('Policy Loss During Training', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Entropy
ax = axes[1, 1]
ent_b = np.array(agent_baseline.entropies)
ent_nb = np.array(agent_no_baseline.entropies)
# Subsample for plotting
step = max(1, len(ent_b) // 500)
ax.plot(ent_b[::step], alpha=0.7, color='blue', label='With Baseline')
ax.plot(ent_nb[::step], alpha=0.7, color='red', label='Without Baseline')
ax.set_xlabel('Step (subsampled)')
ax.set_ylabel('Policy Entropy')
ax.set_title('Exploration (Entropy) Over Training', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('REINFORCE: With vs Without Baseline', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Deep Dive: Likelihood Ratio Trick — Properties and Extensions

### The Score Function

The gradient $\nabla_\theta \log \pi_\theta(a|s)$ is called the **score function** in statistics. It has remarkable properties.

### Property 1: Zero Mean

$$\mathbb{E}_{a \sim \pi_\theta(\cdot|s)}\left[\nabla_\theta \log \pi_\theta(a|s)\right] = \sum_a \pi_\theta(a|s) \frac{\nabla_\theta \pi_\theta(a|s)}{\pi_\theta(a|s)} = \sum_a \nabla_\theta \pi_\theta(a|s) = \nabla_\theta \underbrace{\sum_a \pi_\theta(a|s)}_{=1} = \mathbf{0}$$

**Consequence:** The policy gradient with any state-dependent baseline $b(s)$ is unbiased:

$$\mathbb{E}\left[\nabla_\theta \log \pi_\theta(a|s) \cdot b(s)\right] = b(s) \cdot \mathbb{E}\left[\nabla_\theta \log \pi_\theta(a|s)\right] = \mathbf{0}$$

### Property 2: Covariance Equals Fisher Information

$$\text{Cov}_{a \sim \pi_\theta}\left[\nabla_\theta \log \pi_\theta(a|s)\right] = \mathbb{E}\left[\nabla_\theta \log \pi_\theta \cdot (\nabla_\theta \log \pi_\theta)^T\right] = \mathbf{F}_\theta(s)$$

This follows immediately from Property 1 (zero mean), since $\text{Cov}[\mathbf{X}] = \mathbb{E}[\mathbf{X}\mathbf{X}^T] - \mathbb{E}[\mathbf{X}]\mathbb{E}[\mathbf{X}]^T = \mathbb{E}[\mathbf{X}\mathbf{X}^T]$ when $\mathbb{E}[\mathbf{X}] = \mathbf{0}$.

### Property 3: Score Function for Common Policies

**Softmax (discrete actions):** $\pi_\theta(a|s) = \frac{e^{\theta^T \phi(s,a)}}{\sum_{a'} e^{\theta^T \phi(s,a')}}$

$$\nabla_\theta \log \pi_\theta(a|s) = \phi(s,a) - \mathbb{E}_{a' \sim \pi_\theta}[\phi(s,a')]$$

**Gaussian (continuous actions):** $\pi_\theta(a|s) = \mathcal{N}(\mu_\theta(s), \sigma_\theta^2(s))$

$$\nabla_{\mu} \log \pi = \frac{a - \mu}{\sigma^2}, \qquad \nabla_{\sigma} \log \pi = \frac{(a - \mu)^2 - \sigma^2}{\sigma^3}$$

### The Reward-to-Go Improvement (Causality)

The REINFORCE estimator can be tightened using **causality**: action $a_t$ cannot affect past rewards $r_0, \ldots, r_{t-1}$.

**Theorem:** $\nabla_\theta J = \mathbb{E}\left[\sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$ where $G_t = \sum_{k=t}^{T-1} \gamma^{k-t} r_k$.

**Proof:** Starting from $\nabla_\theta J = \mathbb{E}\left[\sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot R(\tau)\right]$, write $R(\tau) = \sum_{t'=0}^{T-1} \gamma^{t'} r_{t'}$ and split into past and future:

$$\mathbb{E}\left[\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot \sum_{t'=0}^{t-1} \gamma^{t'} r_{t'}\right] = 0$$

because $\sum_{t'<t} \gamma^{t'} r_{t'}$ depends only on $s_0, a_0, \ldots, s_{t-1}, a_{t-1}$, and:

$$\mathbb{E}_{a_t}\left[\nabla_\theta \log \pi_\theta(a_t|s_t)\right] = 0$$

by Property 1. So only the future rewards $G_t = \sum_{k=t}^{T-1} \gamma^{k-t} r_k$ survive. $\blacksquare$

**Variance reduction:** Using $G_t$ instead of $R(\tau)$ removes the variance contribution of past rewards, which can be substantial for long episodes.

In [ ]:
# Evaluate trained agent

def evaluate_policy_agent(agent, env, n_eval=4):
    """Evaluate the REINFORCE agent."""
    fig, axes = plt.subplots(n_eval, 3, figsize=(12, 4 * n_eval))

    for i in range(n_eval):
        state = env.reset()
        initial_psnr = env.prev_psnr
        initial_img = state.copy()
        actions_taken = []

        for t in range(env.max_steps):
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                probs = agent.policy_net(state_tensor)
            action = probs.argmax(dim=1).item()
            actions_taken.append(env.ACTIONS[action])
            state, _, done, info = env.step(action)
            if done:
                break

        axes[i, 0].imshow(env.target.transpose(1, 2, 0))
        axes[i, 0].set_title('Target', fontsize=10)
        axes[i, 1].imshow(initial_img.transpose(1, 2, 0))
        axes[i, 1].set_title(f'Input ({initial_psnr:.1f} dB)', fontsize=10)
        axes[i, 2].imshow(state.transpose(1, 2, 0))
        axes[i, 2].set_title(f'Enhanced ({info["psnr"]:.1f} dB)', fontsize=10)

        for ax in axes[i]:
            ax.axis('off')

    plt.suptitle('REINFORCE Agent: Image Enhancement', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


env = ImageEnhancementEnv(image_size=48, max_steps=8)
evaluate_policy_agent(agent_baseline, env, n_eval=3)

## Deep Dive: Convergence Theory of REINFORCE

### Convergence Rate

**Theorem (Williams, 1992; later refined):** Under standard regularity conditions (bounded rewards, Lipschitz policy gradients, bounded score function), REINFORCE with step size $\alpha_k$ satisfying $\sum_k \alpha_k = \infty$ and $\sum_k \alpha_k^2 < \infty$ converges to a **local optimum** of $J(\theta)$.

With constant step size $\alpha$ and $N$ trajectories per update:

$$\min_{k \leq K} \|\nabla_\theta J(\theta_k)\|^2 \leq O\left(\frac{1}{\sqrt{K}} + \frac{\sigma^2}{N}\right)$$

where $\sigma^2$ is the variance of the gradient estimator.

### Sample Complexity

To reach an $\epsilon$-approximate stationary point ($\|\nabla J\| \leq \epsilon$):

$$\text{Total trajectories} = O\left(\frac{1}{\epsilon^4}\right) \quad \text{(without baseline)}$$

$$\text{Total trajectories} = O\left(\frac{1}{\epsilon^2}\right) \quad \text{(with optimal baseline, under favorable conditions)}$$

**Derivation sketch:** The convergence rate is governed by the variance of the gradient estimator. Without a baseline, $\text{Var}[\hat{g}] = O(\sigma^2/N)$ where $\sigma^2 \propto \mathbb{E}[G_t^2]$. The mean return $\mathbb{E}[G_t]$ contributes to variance but not to the gradient signal — subtracting it (baseline) removes this component.

### Why Only Local Optima?

The policy optimization landscape $J(\theta)$ is generally **non-convex**. Even for simple MDPs with softmax policies:

$$J(\theta) = \sum_s d^{\pi_\theta}(s) \sum_a \pi_\theta(a|s) Q^{\pi_\theta}(s,a)$$

Both $d^{\pi_\theta}$ and $Q^{\pi_\theta}$ depend on $\theta$, creating a complex, non-convex objective. However:

1. For **tabular softmax** policies, all local optima are global (Agarwal et al., 2020)
2. For **neural network** policies, spurious local optima can exist
3. In practice, entropy regularization helps avoid poor local optima

### Comparison with Value-Based Methods

| Property | REINFORCE | DQN |
|----------|-----------|-----|
| Convergence guarantee | Local optimum | Optimal Q* (tabular) |
| Sample complexity | $O(1/\epsilon^4)$ | $O(1/\epsilon^2)$ (tabular) |
| Continuous actions | Natural | Requires discretization |
| Stochastic policies | Yes | No (deterministic) |
| On-policy data | Required | Can use replay buffer |

The high sample complexity of REINFORCE motivates **Actor-Critic** methods (lower variance via bootstrapping) and **PPO** (data reuse via importance sampling).

---

## 8. Summary

### Policy Gradient Theorem (Recap)

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot (G_t - b(s_t))\right]$$

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Log-derivative trick** | $\nabla p = p \cdot \nabla \log p$ enables sampling-based gradient estimation |
| **Causality** | Use reward-to-go $G_t$ instead of full return $R(\tau)$ |
| **Baseline** | Subtracting $b(s)$ reduces variance without adding bias |
| **Value baseline** | $b(s) = V^\pi(s)$ gives advantage $A^\pi(s,a)$ — near-optimal variance reduction |
| **Entropy bonus** | Adding $H(\pi)$ to loss encourages exploration |

### Advantages of Policy Gradient over DQN

- Can handle **continuous action spaces** naturally
- Learns **stochastic policies** (useful for exploration)
- Smoother optimization landscape (no max operation)
- Directly optimizes the performance objective

### Limitations

- High variance (mitigated by baselines)
- Sample inefficient (on-policy — cannot reuse old data)
- Can converge to local optima

### Next: Actor-Critic (Module 6.3)

Combines policy gradient with learned value function, using **bootstrapped** returns instead of Monte Carlo for lower variance.